In [3]:
# Connecting Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Copy dataset from GDrive
import os
import shutil

# Google Drive dataset
source_root = "/content/drive/MyDrive/DeepLearning/Plant_Leaf_Disease_Detector-CNN/dataset"

# Local Colab destination
destination_root = "/content/dataset"

# Classes you want
selected_classes = [
    "Potato___Early_blight",
    "Potato___healthy",
    "Potato___Late_blight"
]

# Splits
splits = ["train", "val", "test"]

for split in splits:
    for class_name in selected_classes:
        
        source = os.path.join(source_root, split, class_name)
        destination = os.path.join(destination_root, split, class_name)

        if os.path.exists(source):
            shutil.copytree(source, destination, dirs_exist_ok=True)
            print(f"Copied: {split}/{class_name}")
        else:
            print(f"Not found: {source}")

print("\nDone!")

Copied: train/Potato___Early_blight
Copied: train/Potato___healthy
Copied: train/Potato___Late_blight
Copied: val/Potato___Early_blight
Copied: val/Potato___healthy
Copied: val/Potato___Late_blight
Copied: test/Potato___Early_blight
Copied: test/Potato___healthy
Copied: test/Potato___Late_blight

Done!


In [5]:
# Importing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from tensorflow.keras import layers, models 

In [6]:
IMAGE_SIZE = 256
BATCH_SIZE = 32
CHANNELS = 3

In [7]:
# Reading and preprocessing the dataset 
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
  '/content/dataset/train',
  shuffle=True,
  image_size=(IMAGE_SIZE, IMAGE_SIZE),
  batch_size = BATCH_SIZE
)
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
  '/content/dataset/test',
  shuffle=False,
  image_size=(IMAGE_SIZE, IMAGE_SIZE),
  batch_size = BATCH_SIZE
)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
  '/content/dataset/val',
  shuffle=False,
  image_size=(IMAGE_SIZE, IMAGE_SIZE),
  batch_size = BATCH_SIZE
)

Found 1506 files belonging to 3 classes.
Found 216 files belonging to 3 classes.
Found 430 files belonging to 3 classes.


In [8]:
class_names = train_ds.class_names
print(class_names)

['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy']


In [9]:
# Optimize dataset for fast training
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size = tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size = tf.data.AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size = tf.data.AUTOTUNE)

In [10]:
# Building model
input_shape = (IMAGE_SIZE, IMAGE_SIZE, CHANNELS)
model = models.Sequential([
  
  layers.Input(shape=input_shape),

  # Rescaling, and Augementation the images
  layers.Rescaling(1.0/255),
  layers.RandomFlip('horizontal_and_vertical'),
  layers.RandomRotation(0.2),

  layers.Conv2D(32, (3,3), activation='relu'),
  layers.MaxPool2D((2,2)),

  layers.Conv2D(64, (3,3), activation='relu'),
  layers.MaxPool2D((2,2)),

  layers.Conv2D(64, (3,3), activation='relu'),
  layers.MaxPool2D((2,2)),

  layers.Conv2D(64, (3,3), activation='relu'),
  layers.MaxPool2D((2,2)),

  layers.Conv2D(64, (3,3), activation='relu'),
  layers.MaxPool2D((2,2)),

  layers.Conv2D(64, (3,3), activation='relu'),
  layers.MaxPool2D((2,2)),

  layers.Flatten(),
  
  layers.Dense(64, activation='relu'),
  layers.Dense(3, activation='softmax')
])

In [11]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 256, 256, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 60, 60, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 12, 12, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 4, 4, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 2, 2, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 183,747 (717.76 KB)

 Trainable params: 183,747 (717.76 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
# Compile the model
model.compile(
  optimizer='adam',
  loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
  metrics=['accuracy']
)

In [ ]:
# Train the model
history = model.fit(train_ds, epochs=50, batch_size=BATCH_SIZE, verbose=1, validation_data=val_ds)

Epoch 1/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 12s 83ms/step - accuracy: 0.4980 - loss: 0.9273 - val_accuracy: 0.4930 - val_loss: 0.8826
Epoch 2/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 3s 59ms/step - accuracy: 0.5584 - loss: 0.8750 - val_accuracy: 0.6884 - val_loss: 0.7728
Epoch 3/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 3s 56ms/step - accuracy: 0.7497 - loss: 0.5998 - val_accuracy: 0.7977 - val_loss: 0.5271
Epoch 4/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 3s 56ms/step - accuracy: 0.8174 - loss: 0.4178 - val_accuracy: 0.8512 - val_loss: 0.3621
Epoch 5/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 3s 56ms/step - accuracy: 0.8699 - loss: 0.3156 - val_accuracy: 0.8674 - val_loss: 0.3093
Epoch 6/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - accuracy: 0.8831 - loss: 0.2879 - val_accuracy: 0.8930 - val_loss: 0.2657
Epoch 7/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - accuracy: 0.8984 - loss: 0.2669 - val_accuracy: 0.8791 - val_loss: 0.3136
Epoch 8/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 3s 56ms/step - accuracy: 0.8811 - loss: 0.2780 - val_accuracy: 0.8558 - 

In [ ]:
# Evaluate the model
score = model.evaluate(test_ds)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.9491 - loss: 0.2780  


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']